In [0]:
base_path = "/Volumes/workspace/default_capstone/capstone_files/"

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "sales.csv")
)

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "products.csv")
)

display(sales_df)
display(products_df)

sale_id,store_id,product_id,quantity,price,cost,discount
1,101,201,10,500,300,20
2,101,202,5,1000,650,50
3,102,201,8,500,300,25
4,102,203,4,1500,1000,100
5,103,202,12,1000,650,75
6,103,203,3,1500,1000,50
7,101,203,6,1500,1000,0
8,102,202,7,1000,650,30
9,103,201,9,500,300,10
10,101,201,5,500,300,0


product_id,product_name,category
201,Laptop,Electronics
202,Headphones,Accessories
203,Keyboard,Accessories


In [0]:
print("SALES DATA")
sales_df.printSchema()

print("PRODUCTS DATA")
products_df.printSchema()

SALES DATA
root
 |-- sale_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- cost: integer (nullable = true)
 |-- discount: integer (nullable = true)

PRODUCTS DATA
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [0]:
display(sales_df.limit(5))
display(products_df.limit(5))

sale_id,store_id,product_id,quantity,price,cost,discount
1,101,201,10,500,300,20
2,101,202,5,1000,650,50
3,102,201,8,500,300,25
4,102,203,4,1500,1000,100
5,103,202,12,1000,650,75


product_id,product_name,category
201,Laptop,Electronics
202,Headphones,Accessories
203,Keyboard,Accessories


In [0]:
from pyspark.sql.functions import *

product_sales = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount")
    )
)

product_sales = (
    product_sales
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "profit_margin",
        when(
            col("net_revenue") != 0,
            round(col("profit") / col("net_revenue") * 100, 2)
        ).otherwise(0)
    )
)

display(product_sales)

product_id,total_quantity_sold,gross_revenue,total_cost,total_discount,net_revenue,profit,profit_margin
201,32,16000,9600,55,15945,6345,39.79
202,24,24000,15600,155,23845,8245,34.58
203,13,19500,13000,150,19350,6350,32.82


In [0]:
final_product_metrics = (
    product_sales
    .join(products_df, on="product_id", how="inner")
    .select(
        "product_id",
        "product_name",
        "category",
        "total_quantity_sold",
        "gross_revenue",
        "total_discount",
        "net_revenue",
        "total_cost",
        "profit",
        "profit_margin"
    )
)

display(final_product_metrics)

product_id,product_name,category,total_quantity_sold,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin
201,Laptop,Electronics,32,16000,55,15945,9600,6345,39.79
202,Headphones,Accessories,24,24000,155,23845,15600,8245,34.58
203,Keyboard,Accessories,13,19500,150,19350,13000,6350,32.82


In [0]:
from pyspark.sql.functions import *

product_sales = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount"),
        count("sale_id").alias("total_orders"),
        avg("price").alias("average_selling_price"),
        avg("cost").alias("average_cost")
    )
)

product_sales = (
    product_sales
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "profit_margin",
        round(col("profit") / col("net_revenue") * 100, 2)
    )
)

final_product_metrics = (
    product_sales
    .join(products_df, "product_id", "inner")
    .select(
        "product_id",
        "product_name",
        "category",
        "total_orders",
        "total_quantity_sold",
        "average_selling_price",
        "average_cost",
        "gross_revenue",
        "total_discount",
        "net_revenue",
        "total_cost",
        "profit",
        "profit_margin"
    )
)

display(final_product_metrics)

product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82


In [0]:
category_metrics = (
    final_product_metrics
    .groupBy("category")
    .agg(
        sum("total_orders").alias("total_orders"),
        sum("total_quantity_sold").alias("total_quantity_sold"),
        sum("gross_revenue").alias("gross_revenue"),
        sum("total_discount").alias("total_discount"),
        sum("net_revenue").alias("net_revenue"),
        sum("total_cost").alias("total_cost"),
        sum("profit").alias("total_profit")
    )
)

category_metrics = (
    category_metrics
    .withColumn(
        "profit_margin",
        round(col("total_profit") / col("net_revenue") * 100, 2)
    )
)

display(category_metrics)

category,total_orders,total_quantity_sold,gross_revenue,total_discount,net_revenue,total_cost,total_profit,profit_margin
Electronics,4,32,16000,55,15945,9600,6345,39.79
Accessories,6,37,43500,305,43195,28600,14595,33.79


In [0]:
final_product_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_product_metrics")

In [0]:
category_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_category_metrics")

In [0]:
%sql
SELECT
    product_id,
    product_name,
    category,
    total_quantity_sold,
    net_revenue,
    profit
FROM workspace.default.retail_product_metrics
ORDER BY total_quantity_sold DESC
LIMIT 3;

product_id,product_name,category,total_quantity_sold,net_revenue,profit
201,Laptop,Electronics,32,15945,6345
202,Headphones,Accessories,24,23845,8245
203,Keyboard,Accessories,13,19350,6350


In [0]:
%sql
SELECT *
FROM workspace.default.retail_category_metrics
ORDER BY total_profit DESC;

category,total_orders,total_quantity_sold,gross_revenue,total_discount,net_revenue,total_cost,total_profit,profit_margin
Accessories,6,37,43500,305,43195,28600,14595,33.79
Electronics,4,32,16000,55,15945,9600,6345,39.79


In [0]:
print("PRODUCT METRICS")
display(spark.table("workspace.default.retail_product_metrics"))

print("CATEGORY METRICS")
display(spark.table("workspace.default.retail_category_metrics"))

PRODUCT METRICS


product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82


CATEGORY METRICS


category,total_orders,total_quantity_sold,gross_revenue,total_discount,net_revenue,total_cost,total_profit,profit_margin
Electronics,4,32,16000,55,15945,9600,6345,39.79
Accessories,6,37,43500,305,43195,28600,14595,33.79


In [0]:
new_products = spark.createDataFrame(
    [
        (204, "Mouse", "Accessories"),
        (205, "Monitor", "Electronics"),
        (206, "Printer", "Electronics")
    ],
    ["product_id", "product_name", "category"]
)

products_df = products_df.unionByName(new_products)

display(products_df)

product_id,product_name,category
201,Laptop,Electronics
202,Headphones,Accessories
203,Keyboard,Accessories
204,Mouse,Accessories
205,Monitor,Electronics
206,Printer,Electronics


In [0]:
new_sales = spark.createDataFrame(
    [
        (6, 104, 204, 15, 500, 250, 30),
        (7, 104, 205, 6, 8000, 5500, 200),
        (8, 105, 206, 4, 6000, 4000, 100),
        (9, 105, 201, 3, 500, 300, 10),
        (10, 106, 202, 8, 1000, 650, 40),
        (11, 106, 204, 10, 500, 250, 20)
    ],
    ["sale_id", "store_id", "product_id", "quantity",
     "price", "cost", "discount"]
)

sales_df = sales_df.unionByName(new_sales)

display(sales_df)

sale_id,store_id,product_id,quantity,price,cost,discount
1,101,201,10,500,300,20
2,101,202,5,1000,650,50
3,102,201,8,500,300,25
4,102,203,4,1500,1000,100
5,103,202,12,1000,650,75
6,103,203,3,1500,1000,50
7,101,203,6,1500,1000,0
8,102,202,7,1000,650,30
9,103,201,9,500,300,10
10,101,201,5,500,300,0


In [0]:
print("Total sales records:", sales_df.count())
print("Total products:", products_df.count())

Total sales records: 16
Total products: 6


In [0]:
from pyspark.sql.functions import *

large_sales = (
    spark.range(17, 50001)
    .withColumnRenamed("id", "sale_id")
    .withColumn("store_id", (col("sale_id") % 10 + 101).cast("int"))
    .withColumn("product_id", (col("sale_id") % 6 + 201).cast("int"))
    .withColumn("quantity", (col("sale_id") % 15 + 1).cast("int"))
    .withColumn(
        "price",
        when(col("product_id") == 201, 500)
        .when(col("product_id") == 202, 1000)
        .when(col("product_id") == 203, 1500)
        .when(col("product_id") == 204, 500)
        .when(col("product_id") == 205, 8000)
        .otherwise(6000)
    )
    .withColumn(
        "cost",
        when(col("product_id") == 201, 300)
        .when(col("product_id") == 202, 650)
        .when(col("product_id") == 203, 1000)
        .when(col("product_id") == 204, 250)
        .when(col("product_id") == 205, 5500)
        .otherwise(4000)
    )
    .withColumn("discount", (col("quantity") * 5).cast("int"))
    .select(
        "sale_id", "store_id", "product_id",
        "quantity", "price", "cost", "discount"
    )
)

sales_df = (
    sales_df.unionByName(large_sales)
    .dropDuplicates(["sale_id"])
)

display(sales_df)

sale_id,store_id,product_id,quantity,price,cost,discount
9,103,201,9,500,300,10
3,102,201,8,500,300,25
4,102,203,4,1500,1000,100
6,103,203,3,1500,1000,50
2,101,202,5,1000,650,50
7,101,203,6,1500,1000,0
5,103,202,12,1000,650,75
8,102,202,7,1000,650,30
1,101,201,10,500,300,20
10,101,201,5,500,300,0


In [0]:
print("Total Sales Records:", sales_df.count())
print("Total Products:", products_df.count())

Total Sales Records: 49995
Total Products: 6


In [0]:
from pyspark.sql.functions import *

new_products = (
    spark.range(207, 701)
    .withColumnRenamed("id", "product_id")
    .withColumn(
        "product_name",
        concat(lit("Product_"), col("product_id"))
    )
    .withColumn(
        "category",
        when(col("product_id") % 4 == 0, "Electronics")
        .when(col("product_id") % 4 == 1, "Accessories")
        .when(col("product_id") % 4 == 2, "Furniture")
        .otherwise("Stationery")
    )
    .select("product_id", "product_name", "category")
)

products_df = products_df.unionByName(new_products)

display(products_df)

product_id,product_name,category
201,Laptop,Electronics
202,Headphones,Accessories
203,Keyboard,Accessories
204,Mouse,Accessories
205,Monitor,Electronics
206,Printer,Electronics
207,Product_207,Stationery
208,Product_208,Electronics
209,Product_209,Accessories
210,Product_210,Furniture


In [0]:
store_metrics = (
    sales_df
    .groupBy("store_id")
    .agg(
        count("sale_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount")
    )
)

store_metrics = (
    store_metrics
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "profit_margin",
        round(col("profit") / col("net_revenue") * 100, 2)
    )
)

display(store_metrics.orderBy(desc("net_revenue")))

store_id,total_orders,total_quantity,gross_revenue,total_cost,total_discount,net_revenue,profit,profit_margin
109,4999,44986,212417000,145276400,224930,212192070,66915670,31.54
110,4999,49985,166605000,109542750,249925,166355075,56812325,34.15
101,5003,30020,162465500,111308550,150040,162315460,51006910,31.42
102,5001,35005,129132000,85060250,175085,128956915,43896665,34.04
103,5001,40008,124971000,84812900,200055,124770945,39958045,32.03
104,4998,44982,116620000,76802600,224910,116395090,39592490,34.02
105,4998,49980,112455000,75803000,249900,112205100,36402100,32.44
106,4999,29998,79140000,52314900,149960,78990040,26675140,33.77
107,4998,34986,62475000,41816600,174930,62300070,20483470,32.88
108,4999,39987,54163000,34081700,199935,53963065,19881365,36.84


In [0]:
store_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_store_metrics")

In [0]:
%sql
SELECT
    store_id,
    total_orders,
    total_quantity,
    net_revenue,
    profit,
    profit_margin
FROM workspace.default.retail_store_metrics
ORDER BY net_revenue DESC
LIMIT 5;

store_id,total_orders,total_quantity,net_revenue,profit,profit_margin
109,4999,44986,212192070,66915670,31.54
110,4999,49985,166355075,56812325,34.15
101,5003,30020,162315460,51006910,31.42
102,5001,35005,128956915,43896665,34.04
103,5001,40008,124770945,39958045,32.03


In [0]:
top_profitable_products = (
    final_product_metrics
    .orderBy(desc("profit"))
    .limit(10)
)

display(top_profitable_products)

product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79


In [0]:
from pyspark.sql.window import Window

product_window = Window.orderBy(desc("total_quantity_sold"))

product_ranking = (
    final_product_metrics
    .withColumn("sales_rank", dense_rank().over(product_window))
)

display(product_ranking)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin,sales_rank
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79,1
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58,2
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82,3


In [0]:
product_ranking.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_product_ranking")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT *
FROM workspace.default.retail_product_ranking
ORDER BY sales_rank
LIMIT 10;

product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin,sales_rank
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79,1
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58,2
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82,3


In [0]:
discount_analysis = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("discount").alias("total_discount"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        count("sale_id").alias("total_orders")
    )
    .withColumn(
        "net_revenue",
        col("gross_revenue") - col("total_discount")
    )
    .withColumn(
        "profit",
        col("net_revenue") - col("total_cost")
    )
    .withColumn(
        "discount_percentage",
        round(col("total_discount") / col("gross_revenue") * 100, 2)
    )
)

display(discount_analysis)

product_id,total_discount,gross_revenue,total_cost,total_orders,net_revenue,profit,discount_percentage
201,291625,29173000,17503800,8335,28881375,11377575,1.0
203,375030,112483500,74989000,8334,112108470,37119470,0.33
202,333380,66669000,43334850,8334,66335620,23000770,0.5
204,291570,29160000,14580000,8331,28868430,14288430,1.0
205,333200,533120000,366520000,8330,532786800,166266800,0.06
206,374865,449838000,299892000,8331,449463135,149571135,0.08


In [0]:
discount_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_discount_analysis")

In [0]:
%sql
SELECT
    product_id,
    total_discount,
    gross_revenue,
    net_revenue,
    profit,
    discount_percentage
FROM workspace.default.retail_discount_analysis
ORDER BY total_discount DESC
LIMIT 10;

product_id,total_discount,gross_revenue,net_revenue,profit,discount_percentage
203,375030,112483500,112108470,37119470,0.33
206,374865,449838000,449463135,149571135,0.08
202,333380,66669000,66335620,23000770,0.5
205,333200,533120000,532786800,166266800,0.06
201,291625,29173000,28881375,11377575,1.0
204,291570,29160000,28868430,14288430,1.0


In [0]:
from pyspark.sql.functions import *

sales_with_date = (
    sales_df
    .withColumn(
        "sale_date",
        date_add(
            to_date(lit("2026-10-01")),
            (col("sale_id") % 30).cast("int")
        )
    )
)

daily_sales = (
    sales_with_date
    .groupBy("sale_date")
    .agg(
        count("sale_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum("discount").alias("total_discount")
    )
    .withColumn(
        "net_revenue",
        col("gross_revenue") - col("total_discount")
    )
    .orderBy("sale_date")
)

display(daily_sales)

sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue
2026-10-01,1666,1666,833000,8330,824670
2026-10-02,1667,3342,3337000,16680,3320320
2026-10-03,1667,5003,7502000,25040,7476960
2026-10-04,1667,6672,3336000,33345,3302655
2026-10-05,1667,8334,66646000,41750,66604250
2026-10-06,1667,10008,59988000,50055,59937945
2026-10-07,1667,11665,5835500,58360,5777140
2026-10-08,1667,13334,13337000,66640,13270360
2026-10-09,1667,15001,22498000,75000,22423000
2026-10-10,1667,16669,8334500,83310,8251190


In [0]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_daily_sales")

In [0]:
%sql
SELECT *
FROM workspace.default.retail_daily_sales
ORDER BY net_revenue DESC
LIMIT 5;

sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue
2026-10-29,1666,23324,186592000,116620,186475380
2026-10-30,1666,24990,149940000,124950,149815050
2026-10-11,1667,18331,146610500,91630,146518870
2026-10-12,1667,20002,119957000,99980,119857020
2026-10-23,1666,13328,106624000,66640,106557360


In [0]:
from pyspark.sql.functions import *

quality_report = sales_df.agg(
    count("*").alias("total_records"),
    countDistinct("sale_id").alias("unique_sales"),
    sum(when(col("sale_id").isNull(), 1).otherwise(0)).alias("missing_sale_ids"),
    sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("missing_product_ids"),
    sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("missing_quantities"),
    sum(when(col("price") <= 0, 1).otherwise(0)).alias("invalid_prices"),
    sum(when(col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantities")
)

display(quality_report)

total_records,unique_sales,missing_sale_ids,missing_product_ids,missing_quantities,invalid_prices,invalid_quantities
49995,49995,0,0,0,0,0


In [0]:
duplicate_sales = (
    sales_df
    .groupBy("sale_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_sales)

sale_id,count


In [0]:
low_selling_products = (
    final_product_metrics
    .filter(col("total_quantity_sold") < 100)
    .orderBy("total_quantity_sold")
)

display(low_selling_products)

product_id,product_name,category,total_orders,total_quantity_sold,average_selling_price,average_cost,gross_revenue,total_discount,net_revenue,total_cost,profit,profit_margin
203,Keyboard,Accessories,3,13,1500.0,1000.0,19500,150,19350,13000,6350,32.82
202,Headphones,Accessories,3,24,1000.0,650.0,24000,155,23845,15600,8245,34.58
201,Laptop,Electronics,4,32,500.0,300.0,16000,55,15945,9600,6345,39.79


In [0]:
low_selling_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_low_selling_products")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

revenue_window = (
    Window
    .orderBy("sale_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_revenue = (
    daily_sales
    .withColumn(
        "cumulative_revenue",
        sum("net_revenue").over(revenue_window)
    )
)

display(running_revenue)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue,cumulative_revenue
2026-10-01,1666,1666,833000,8330,824670,824670
2026-10-02,1667,3342,3337000,16680,3320320,4144990
2026-10-03,1667,5003,7502000,25040,7476960,11621950
2026-10-04,1667,6672,3336000,33345,3302655,14924605
2026-10-05,1667,8334,66646000,41750,66604250,81528855
2026-10-06,1667,10008,59988000,50055,59937945,141466800
2026-10-07,1667,11665,5835500,58360,5777140,147243940
2026-10-08,1667,13334,13337000,66640,13270360,160514300
2026-10-09,1667,15001,22498000,75000,22423000,182937300
2026-10-10,1667,16669,8334500,83310,8251190,191188490


In [0]:
running_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_running_revenue")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT *
FROM workspace.default.retail_running_revenue
ORDER BY sale_date;

sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue,cumulative_revenue
2026-10-01,1666,1666,833000,8330,824670,824670
2026-10-02,1667,3342,3337000,16680,3320320,4144990
2026-10-03,1667,5003,7502000,25040,7476960,11621950
2026-10-04,1667,6672,3336000,33345,3302655,14924605
2026-10-05,1667,8334,66646000,41750,66604250,81528855
2026-10-06,1667,10008,59988000,50055,59937945,141466800
2026-10-07,1667,11665,5835500,58360,5777140,147243940
2026-10-08,1667,13334,13337000,66640,13270360,160514300
2026-10-09,1667,15001,22498000,75000,22423000,182937300
2026-10-10,1667,16669,8334500,83310,8251190,191188490


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

moving_window = (
    Window
    .orderBy("sale_date")
    .rowsBetween(-6, 0)
)

moving_average = (
    daily_sales
    .withColumn(
        "moving_average_revenue",
        round(avg("net_revenue").over(moving_window), 2)
    )
)

display(moving_average)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue,moving_average_revenue
2026-10-01,1666,1666,833000,8330,824670,824670.0
2026-10-02,1667,3342,3337000,16680,3320320,2072495.0
2026-10-03,1667,5003,7502000,25040,7476960,3873983.33
2026-10-04,1667,6672,3336000,33345,3302655,3731151.25
2026-10-05,1667,8334,66646000,41750,66604250,1.6305771E7
2026-10-06,1667,10008,59988000,50055,59937945,2.35778E7
2026-10-07,1667,11665,5835500,58360,5777140,2.103484857E7
2026-10-08,1667,13334,13337000,66640,13270360,2.281280429E7
2026-10-09,1667,15001,22498000,75000,22423000,2.554175857E7
2026-10-10,1667,16669,8334500,83310,8251190,2.565236286E7


In [0]:
moving_average.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_moving_average")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT *
FROM workspace.default.retail_moving_average
ORDER BY sale_date;

sale_date,total_orders,total_quantity,gross_revenue,total_discount,net_revenue,moving_average_revenue
2026-10-01,1666,1666,833000,8330,824670,824670.0
2026-10-02,1667,3342,3337000,16680,3320320,2072495.0
2026-10-03,1667,5003,7502000,25040,7476960,3873983.33
2026-10-04,1667,6672,3336000,33345,3302655,3731151.25
2026-10-05,1667,8334,66646000,41750,66604250,1.6305771E7
2026-10-06,1667,10008,59988000,50055,59937945,2.35778E7
2026-10-07,1667,11665,5835500,58360,5777140,2.103484857E7
2026-10-08,1667,13334,13337000,66640,13270360,2.281280429E7
2026-10-09,1667,15001,22498000,75000,22423000,2.554175857E7
2026-10-10,1667,16669,8334500,83310,8251190,2.565236286E7


In [0]:
from pyspark.sql.functions import *

product_sales = (
    sales_df.groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount"),
        count("sale_id").alias("total_orders"),
        avg("price").alias("average_selling_price"),
        avg("cost").alias("average_cost")
    )
    .withColumn(
        "net_revenue",
        col("gross_revenue") - col("total_discount")
    )
    .withColumn(
        "profit",
        col("net_revenue") - col("total_cost")
    )
    .withColumn(
        "profit_margin",
        when(
            col("net_revenue") != 0,
            round(col("profit") / col("net_revenue") * 100, 2)
        ).otherwise(0)
    )
)

final_product_metrics = (
    products_df
    .join(product_sales, "product_id", "left")
    .fillna(0)
)

display(final_product_metrics)

product_id,product_name,category,total_quantity_sold,gross_revenue,total_cost,total_discount,total_orders,average_selling_price,average_cost,net_revenue,profit,profit_margin
201,Laptop,Electronics,58346,29173000,17503800,291625,8335,500.0,300.0,28881375,11377575,39.39
202,Headphones,Accessories,66669,66669000,43334850,333380,8334,1000.0,650.0,66335620,23000770,34.67
203,Keyboard,Accessories,74989,112483500,74989000,375030,8334,1500.0,1000.0,112108470,37119470,33.11
204,Mouse,Accessories,58320,29160000,14580000,291570,8331,500.0,250.0,28868430,14288430,49.5
205,Monitor,Electronics,66640,533120000,366520000,333200,8330,8000.0,5500.0,532786800,166266800,31.21
206,Printer,Electronics,74973,449838000,299892000,374865,8331,6000.0,4000.0,449463135,149571135,33.28
207,Product_207,Stationery,0,0,0,0,0,0.0,0.0,0,0,0.0
208,Product_208,Electronics,0,0,0,0,0,0.0,0.0,0,0,0.0
209,Product_209,Accessories,0,0,0,0,0,0.0,0.0,0,0,0.0
210,Product_210,Furniture,0,0,0,0,0,0.0,0.0,0,0,0.0
